In [1]:
# from pathlib import Path
# import pandas as pd
# import numpy as np

# PROJECT_ROOT = Path("..").resolve()

# DATA_DIR = PROJECT_ROOT / "data"
# OUTPUTS_DIR = PROJECT_ROOT / "outputs"
# EXPERIMENTS_DIR = OUTPUTS_DIR / "experiments"

# RUN_ID = "latest"  # cambiar si hace falta

# PRED_PATH = EXPERIMENTS_DIR / RUN_ID / "predictions.parquet"
# METRICS_PATH = EXPERIMENTS_DIR / RUN_ID / "metrics.parquet"
# COEFS_PATH = EXPERIMENTS_DIR / RUN_ID / "coefficients.parquet"
# FEATURES_PATH = DATA_DIR / "processed" / "modeling_dataset.parquet"
# FEATURE_METADATA_PATH = DATA_DIR / "processed" / "feature_metadata.csv"

# ID_COL = "id_obs"
# Y_TRUE_COL = "y_true"
# Y_PRED_COL = "y_pred"
# MODEL_COL = "model_name"
# EXPERIMENT_COL = "experiment_id"
# WEIGHT_COL = "sample_weight"

In [ ]:
# # 1. Load backend outputs

# df_pred = pd.read_parquet(PRED_PATH)
# df_models = pd.read_parquet(METRICS_PATH)

# import pyarrow.parquet as pq
# available_cols = pq.read_schema(FEATURES_PATH).names

# feature_cols = [
#     ID_COL,
#     "region",
#     "provincia",
#     "aglomerado",
#     "year",
#     "quarter",
#     WEIGHT_COL,
#     "edad",
#     "sexo",
#     "educacion",
#     "condicion_actividad",
#     "categoria_ocupacional",
#     "formalidad",
#     "beneficios_laborales",
# ]

# available_feature_cols = [c for c in feature_cols if c in available_cols]
# df_features = pd.read_parquet(FEATURES_PATH, columns=available_feature_cols)

# if "residual" not in df_pred.columns:
#     df_pred["residual"] = df_pred[Y_TRUE_COL] - df_pred[Y_PRED_COL]

# df_pred.head(), df_features.head(), df_models.head()

In [ ]:
xx

In [3]:
# 2. Build residual and tail-compression analysis dataframes

MODELS_TO_ANALYZE = [
    "ols",
    "ridge",
    "lasso",
    "hgb",
    "random_forest",
    "mlp",
]

available_models = df_pred[MODEL_COL].unique().tolist()
models_to_use = [m for m in MODELS_TO_ANALYZE if m in available_models]

if not models_to_use:
    models_to_use = available_models

df_eval = (
    df_pred[df_pred[MODEL_COL].isin(models_to_use)]
    .merge(df_features, on=ID_COL, how="left")
    .copy()
)

if WEIGHT_COL not in df_eval.columns:
    df_eval[WEIGHT_COL] = 1.0

df_eval["error"] = df_eval[Y_PRED_COL] - df_eval[Y_TRUE_COL]
df_eval["residual"] = df_eval[Y_TRUE_COL] - df_eval[Y_PRED_COL]
df_eval["abs_error"] = df_eval["error"].abs()
df_eval["sq_error"] = df_eval["error"] ** 2

# Income bins based on true y, within full evaluation sample.
# If y is log income, these are log-income quantiles.
df_eval["income_decile"] = pd.qcut(
    df_eval[Y_TRUE_COL],
    q=10,
    duplicates="drop"
)

df_eval["income_bin_fine"] = pd.qcut(
    df_eval[Y_TRUE_COL],
    q=[0, .01, .05, .10, .25, .50, .75, .90, .95, .99, 1],
    duplicates="drop"
)

def weighted_metrics(g):
    w = g[WEIGHT_COL]
    return pd.Series({
        "n": len(g),
        "w_sum": w.sum(),
        "mean_y_true": wmean(g[Y_TRUE_COL], w),
        "mean_y_pred": wmean(g[Y_PRED_COL], w),
        "mean_error": wmean(g["error"], w),
        "mean_residual": wmean(g["residual"], w),
        "mae": wmean(g["abs_error"], w),
        "rmse": np.sqrt(wmean(g["sq_error"], w)),
        "sd_y_true": g[Y_TRUE_COL].std(),
        "sd_y_pred": g[Y_PRED_COL].std(),
        "p90_y_true": g[Y_TRUE_COL].quantile(.90),
        "p90_y_pred": g[Y_PRED_COL].quantile(.90),
        "p95_y_true": g[Y_TRUE_COL].quantile(.95),
        "p95_y_pred": g[Y_PRED_COL].quantile(.95),
        "p99_y_true": g[Y_TRUE_COL].quantile(.99),
        "p99_y_pred": g[Y_PRED_COL].quantile(.99),
    })

df_global_metrics = (
    df_eval
    .groupby(MODEL_COL)
    .apply(weighted_metrics)
    .reset_index()
)

df_bin_metrics = (
    df_eval
    .groupby([MODEL_COL, "income_bin_fine"], observed=True)
    .apply(weighted_metrics)
    .reset_index()
)

# Stretch y_pred to match mean and std of y_true, model by model.
def add_stretched_predictions(g):
    g = g.copy()
    mu_true = g[Y_TRUE_COL].mean()
    sd_true = g[Y_TRUE_COL].std()
    mu_pred = g[Y_PRED_COL].mean()
    sd_pred = g[Y_PRED_COL].std()
    
    if sd_pred == 0 or pd.isna(sd_pred):
        g["y_pred_stretched"] = np.nan
    else:
        g["y_pred_stretched"] = mu_true + (sd_true / sd_pred) * (g[Y_PRED_COL] - mu_pred)
    
    g["error_stretched"] = g["y_pred_stretched"] - g[Y_TRUE_COL]
    g["residual_stretched"] = g[Y_TRUE_COL] - g["y_pred_stretched"]
    g["abs_error_stretched"] = g["error_stretched"].abs()
    g["sq_error_stretched"] = g["error_stretched"] ** 2
    return g

df_stretched = (
    df_eval
    .groupby(MODEL_COL, group_keys=False)
    .apply(add_stretched_predictions)
)

def weighted_metrics_stretched(g):
    w = g[WEIGHT_COL]
    return pd.Series({
        "n": len(g),
        "mae_original": wmean(g["abs_error"], w),
        "mae_stretched": wmean(g["abs_error_stretched"], w),
        "rmse_original": np.sqrt(wmean(g["sq_error"], w)),
        "rmse_stretched": np.sqrt(wmean(g["sq_error_stretched"], w)),
        "sd_y_true": g[Y_TRUE_COL].std(),
        "sd_y_pred": g[Y_PRED_COL].std(),
        "sd_y_pred_stretched": g["y_pred_stretched"].std(),
        "compression_ratio": g[Y_PRED_COL].std() / g[Y_TRUE_COL].std(),
    })

df_stretch_metrics = (
    df_stretched
    .groupby(MODEL_COL)
    .apply(weighted_metrics_stretched)
    .reset_index()
)

df_eval.head(), df_bin_metrics.head(), df_stretch_metrics.head()

NameError: name 'df_pred' is not defined

In [ ]:
DATA_STATUS = {
    "n_predictions": len(df_pred) if "df_pred" in globals() else None,
    "models": sorted(df_pred[MODEL_COL].unique().tolist()) if "df_pred" in globals() else None,
    "n_features_rows": len(df_features) if "df_features" in globals() else None,
    "feature_columns": df_features.columns.tolist() if "df_features" in globals() else None,
    "metrics_columns": df_models.columns.tolist() if "df_models" in globals() else None,
}

DATA_STATUS

In [ ]:
df_eval
df_bin_metrics
df_global_metrics
df_stretched
df_stretch_metrics